In [1]:
pip install torchattacks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.7
    Uninstalling idna-3.7:
      Successfully uninstalled idna-3.7
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-beam 2.46.0 requires cloudpickle~=2.2.1, but you have cloudpickle 3.0.0 which is incompatible.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.8 which is incompatible.
apache-be

In [2]:
import torch
import numpy as np
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets  # Import datasets here
from typing import List
import os
import torchattacks  
import random
import torch.optim as optim
import torch.nn.functional as F
from itertools import cycle



In [3]:
# Define the arguments manually for use in Jupyter Notebook
batch_size = 128
test_batch_size = 1000
epochs = 50
lr = 1.0
gamma = 0.7
no_cuda = False
dry_run = False
seed = 0
log_interval = 10
save_model = False
pgd_eps = 2.0
pgd_alpha = 0.1
pgd_iter = 100
unlearn_label = 9
unlearn_k = 10  # Adjust based on your requirements
unlearn_lr = 0.001
num_adv_images = None
reg_lamb = 10.0
train_kwargs = {'batch_size': 256}
test_kwargs = {'batch_size': 1024}
naiive_unlearn_kwargs = {'batch_size': 32}
k_arr = [16]
#     k_arr = [1, 16, 64, 128, 256]
D_r_acc = []
D_f_acc = []
D_test_acc = []
case2_D_r = []
case2_D_f = []
case2_D_test = []
args = {
    'batch_size': 128,
    'test_batch_size': 1000,
    'epochs': 15,
    'lr': 1.0,
    'gamma': 0.7,
    'no_cuda': False,
    'dry_run': False,
    'seed': 0,
    'log_interval': 10,
    'save_model': False,
    'pgd_eps': 2.0,
    'pgd_alpha': 0.1,
    'pgd_iter': 100,
    'unlearn_label': 9,
    'unlearn_k': 10,
    'unlearn_lr': 0.001,
    'num_adv_images': None,
    'reg_lamb': 10.0
}

In [4]:
class JointDataset(Dataset):
    """Characterizes a dataset for PyTorch -- this dataset accumulates each task dataset incrementally"""

    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels
        self._len = len(inputs)

    def __len__(self):
        'Denotes the total number of samples'
        return self._len

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

In [5]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    CE = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += CE(output, target)  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    return test_loss, 100. * correct / len(test_loader.dataset)


In [6]:
def adv_attack(args, model, device, train_loader, adversary, unlearn_k, num_adv_images = None):
    model.eval()
    
    attacked_image_arr = []
    target_label_arr = []
    
    if num_adv_images == None:

        if 1024 % unlearn_k == 0:
            num_iters = 1024 // unlearn_k
        else:
            num_iters = 1024 // unlearn_k + 1
    else:
        num_iters = num_adv_images

    for i in range (num_iters):

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            attack_label = torch.rand(data.shape[0]).cuda() * 10
            attack_label = attack_label.to(torch.long)
            attack_label = torch.where(attack_label == target, (torch.rand(data.shape[0]).long().cuda()*10 + 10) // 2, attack_label)

            adv_example = adversary(data, attack_label)

            inputs_numpy = adv_example.detach().cpu().numpy()
            labels_numpy = attack_label.cpu().numpy()

            for j in range(inputs_numpy.shape[0]):

                attacked_image_arr.append(inputs_numpy[j])
                target_label_arr.append(labels_numpy[j])
            
            
    return attacked_image_arr, target_label_arr
        


In [7]:
# Code for Resnet 18


__all__ = ["ResNet", "resnet18"]

def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(
        in_planes,
        out_planes,
        kernel_size=3,
        stride=stride,
        padding=dilation,
        groups=groups,
        bias=False,
        dilation=dilation,
    )

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1, norm_layer=None):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=10, zero_init_residual=False, groups=1, width_per_group=64, replace_stride_with_dilation=None, norm_layer=None):
        super(ResNet, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer
        self.inplanes = 64
        self.dilation = 1
        if replace_stride_with_dilation is None:
            replace_stride_with_dilation = [False, False, False]
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)  # CIFAR-10 specific
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation, norm_layer))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups, base_width=self.base_width, dilation=self.dilation, norm_layer=norm_layer))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

def resnet18(pretrained=False, progress=True, device="cpu", **kwargs):
    """Constructs a ResNet-18 model."""
    return ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)


In [8]:


# Check if CUDA is available and set the device accordingly
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

# Update training and testing kwargs if using CUDA
if use_cuda:
    cuda_kwargs = {
        'num_workers': 0,
        'pin_memory': True,
        'shuffle': False
    }
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

# Print the device being used
print(f'Using device: {device}')


Using device: cuda


In [9]:
class NormalizeLayer(torch.nn.Module):
    """Standardize the channels of a batch of images by subtracting the dataset mean
      and dividing by the dataset standard deviation.

      In order to certify radii in original coordinates rather than standardized coordinates, we
      add the Gaussian noise _before_ standardizing, which is why we have standardization be the first
      layer of the classifier rather than as a part of preprocessing as is typical.
      """

    def __init__(self, means: List[float], sds: List[float]):
        """
        :param means: the channel means
        :param sds: the channel standard deviations
        """
        super(NormalizeLayer, self).__init__()
        self.means = torch.tensor(means).cuda()
        self.sds = torch.tensor(sds).cuda()

    def forward(self, input: torch.tensor):
        (batch_size, num_channels, height, width) = input.shape
        means = self.means.repeat((batch_size, height, width, 1)).permute(0, 3, 1, 2)
        sds = self.sds.repeat((batch_size, height, width, 1)).permute(0, 3, 1, 2)
        return (input - means)/sds

In [10]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset1 = datasets.CIFAR10('../data', train=True, download=True, transform=transform)
dataset2 = datasets.CIFAR10('../data', train=False, transform=transform)

for unlearn_k in k_arr:
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    
    unlearn_label = unlearn_label
    train_labels = dataset1.targets
    train_labels = torch.from_numpy(np.array(train_labels))

    indices_k_unlearn = torch.randperm(train_labels.shape[0])[:unlearn_k]
    print('indices_k_unlearn : ', indices_k_unlearn)

    copy_train_labels = train_labels.clone()
    copy_train_labels[indices_k_unlearn] = -10

    indices_other_data = (copy_train_labels != -10).nonzero(as_tuple=False)

    unlearn_dataset = Subset(dataset1, indices_k_unlearn.view(-1,))
    unlearn_loader = torch.utils.data.DataLoader(unlearn_dataset, **naiive_unlearn_kwargs)

    other_dataset = Subset(dataset1, indices_other_data.view(-1,))
    other_loader = torch.utils.data.DataLoader(other_dataset, **test_kwargs)

    cifar_test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

    print('len(unlearn_dataset) : ', len(unlearn_dataset), ' len(other_dataset) : ', len(other_dataset))
    
    model = resnet18().to(device)
    model.load_state_dict(torch.load('/kaggle/input/resnet18modelweights/resnet18.pt'))
    normalize_layer = NormalizeLayer((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    model = torch.nn.Sequential(normalize_layer, model)

    optimizer = optim.SGD(model.parameters(), lr=unlearn_lr, momentum=0.9, weight_decay=1e-4)
        
    origin_params = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

    print ()
    print ('\n Baseline 2: Our Appraoch - using adversarial examples only')

    unlearn_acc = 100
    alpha = 0.0

    model.eval()
    other_loss, other_acc = test(model, device, other_loader)
    unlearn_loss, unlearn_acc = test(model, device, unlearn_loader)
    test_loss, test_acc = test(model, device, cifar_test_loader)
        
    str_list = '\n Before | D_remaining : ' + str(other_acc) +  ', D_forget acc : ' +  str(unlearn_acc)+  ', D_test acc : ' +  str(test_acc)
    print (str_list)

    adversary = torchattacks.PGD(model, eps=pgd_eps, alpha=pgd_alpha, steps=pgd_iter, random_start=True)
    
    adv_images, target_labels = adv_attack(args, model, device, unlearn_loader, adversary, unlearn_k, num_adv_images)

    adv_dataset = JointDataset(adv_images, target_labels)
    adv_loader = torch.utils.data.DataLoader(adv_dataset, **train_kwargs)

    unlearn_acc = 100
    max_iter = 1000
    j = 0
        
    unlearn_loader_cycle = cycle(unlearn_loader)
    CE = nn.CrossEntropyLoss()
        
    while unlearn_acc != 0:
        model.train()

        for i , data in enumerate(zip(adv_loader, unlearn_loader_cycle)):
            model.train()
                
                
            (adv_data, adv_target), (data, target) = data

            optimizer.zero_grad()

            output_adv = model(adv_data.to(device))
            output = model(data.to(device))

            loss_unlearn = -CE(output, target.to(device)) * (data.shape[0] / (adv_data.shape[0] + data.shape[0]))
            loss_adv = CE(output_adv, adv_target.to(device)) * (adv_data.shape[0] / (adv_data.shape[0] + data.shape[0]))

            loss = loss_unlearn + loss_adv

            loss.backward()
            optimizer.step()
                
            model.eval()
            unlearn_loss, unlearn_acc = test(model, device, unlearn_loader)
                
            if unlearn_acc == 0:
                print ('unlearn_acc == 0, Break at j = ', j, ' i = ', i)
                break

            
        j += 1
            
        if max_iter < j:
            break

    model.eval()
    unlearn_loss, unlearn_acc = test(model, device, unlearn_loader)
    other_loss, other_acc = test(model, device, other_loader)
    test_loss, test_acc = test(model, device, cifar_test_loader)
    str_list = '\n After | D_test - D_forget acc : ' + str(other_acc) +  ', D_forget acc : ' +  str(unlearn_acc)+  ', D_test acc : ' +  str(test_acc)
    print (str_list)
            
        
    case2_D_test.append(test_acc)
    case2_D_r.append(other_acc)
    case2_D_f.append(unlearn_acc)


100%|██████████| 170498071/170498071 [00:12<00:00, 13909026.93it/s]


Extracting ../data/cifar-10-python.tar.gz to ../data
indices_k_unlearn :  tensor([36044, 49165, 37807, 11341,  6091, 16904, 36293, 28026, 24681, 17849,
        49031, 20152, 23932, 33744,  8628,  3486])
len(unlearn_dataset) :  16  len(other_dataset) :  49984


/tmp/ipykernel_31/2064932005.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('/kaggle/input/resnet18modelweights/resnet18.pt'))




 Baseline 2: Our Appraoch - using adversarial examples only

 Before | D_remaining : 99.59987195902688, D_forget acc : 100.0, D_test acc : 92.59
unlearn_acc == 0, Break at j =  11  i =  1

 After | D_test - D_forget acc : 13.078185019206146, D_forget acc : 0.0, D_test acc : 13.12
